<img src="https://elementos.entornos.net/clientes/ISPC/ispc.png" width="350" height="200">

#**TECNICATURA SUPERIOR EN CIENCIAS DE DATOS E INTELIGENCIA ARTIFICIAL**

##**"PROCESAMIENTO DEL HABLA"**

TERCERR AÑO - COHORTE 2024


---

## **APLICACIÓN PRÁCTICA CON PYTHON**

---

## Docente:
Rubén Emmanuel GIUDICE BOJANICH

## Estudiantes:


*  Allende Olmedo Nicolás
*  Direni Carlos
*  García Carlos
*  Guaraz Emanuel
*  Moreno Raúl
*  Testa Andrea Paola
*  Villalba Valeria Nieves


Mayo de 2026

#**EVIDENCIA 2 - EJE II**

##Modelado Predictivo y Clasificación de Audio

Entorno: Google Colab | Dataset: ESC-50 (Sonidos Ambientales) | Duración: 2.5-3 horas

**Instrucciones para el alumno:**

Este notebook es secuencial y autocontenido. Ejecuta cada celda en orden. Cada línea está comentada para explicar el qué, por qué y cómo. Al finalizar, responde las actividades en celdas Markdown y exporta el notebook + informe PDF.

CELDA 0: Configuración del Entorno

Descripción: Instala dependencias específicas para ML de audio, importa librerías, configura estilos visuales y fija semillas para reproducibilidad. Incluye instalación de datasets y soundfile para carga robusta de audio.

In [ ]:
# 📥 Instala librerías adicionales para carga de datasets y procesamiento de audio
!pip install datasets soundfile scikit-learn ipywidgets -q

# 📚 Importa NumPy para operaciones numéricas y manejo de matrices
import numpy as np
# 📚 Importa Matplotlib para generación de gráficos 2D
import matplotlib.pyplot as plt
# 📚 Importa Librosa, librería estándar para análisis y extracción de features de audio
import librosa
# 📚 Importa submódulo de visualización de Librosa
import librosa.display
# 📚 Importa Pandas para manipulación de datos tabulares
import pandas as pd
# 📚 Importa Seaborn para visualizaciones estadísticas estéticas
import seaborn as sns
# 📚 Importa módulos de scikit-learn para modelado y evaluación
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# 📚 Importa Hugging Face Datasets para carga segura y reproducible de datos
from datasets import load_dataset
# 📚 Importa widgets interactivos para exploración guiada
import ipywidgets as widgets
from IPython.display import display

# ⚙️ Fija semilla aleatoria para garantizar reproducibilidad entre ejecuciones
np.random.seed(42)
# 🎨 Configura tamaño por defecto de figuras
plt.rcParams['figure.figsize'] = (10, 5)
# 🎨 Establece tamaño base de fuente en gráficos
plt.rcParams['font.size'] = 11
# 🎨 Activa cuadrícula de fondo por defecto
plt.rcParams['axes.grid'] = True
# 🎨 Aplica estilo visual limpio y profesional
plt.style.use('seaborn-v0_8-whitegrid')

# ✅ Confirma configuración exitosa
print("✅ Entorno configurado. Librerías de ML y audio listas.")

CELDA 1: Carga del Nuevo Dataset (ESC-50)

Descripción: Carga un subconjunto del dataset ESC-50 (Environmental Sound Classification). Filtra 4 clases representativas (dog, rain, sea, wind) para mantener el laboratorio ágil. Convierte los arrays de audio a formato compatible con Librosa y organiza metadatos.

In [ ]:
# 🌐 Carga dataset ESC-50 desde Hugging Face (split de entrenamiento completo)
print("⏳ Cargando dataset ESC-50...")
dataset = load_dataset("ashraq/esc50", split="train")

# 🏷️ Define clases objetivo para este laboratorio (subset para agilidad)
target_classes = ["dog", "rain", "sea", "wind"]

# 🔍 Filtra solo las clases de interés para reducir tiempo de procesamiento
filtered_data = [item for item in dataset if item["category"] in target_classes]

# 📦 Inicializa listas para almacenar audios, sample rates y etiquetas
audios, sample_rates, labels = [], [], []

# 🔄 Itera sobre cada muestra filtrada
for item in filtered_data:
    # 🎧 Extrae array de audio y frecuencia de muestreo del item del dataset
    audio_array = item["audio"]["array"]
    sr = item["audio"]["sampling_rate"]
    #  Almacena audio, SR y etiqueta en listas separadas
    audios.append(audio_array)
    sample_rates.append(sr)
    labels.append(item["category"])

# 📊 Convierte listas a DataFrame para manipulación tabular y trazabilidad
df_raw = pd.DataFrame({
    "audio": audios,          # Array NumPy con amplitudes normalizadas
    "sr": sample_rates,       # Frecuencia de muestreo original (típicamente 44100 Hz)
    "label": labels           # Etiqueta categórica del sonido ambiental
})

# ✅ Imprime resumen del dataset cargado
print(f"📊 Dataset cargado: {len(df_raw)} muestras")
print(f"🏷️ Distribución por clase:\n{df_raw['label'].value_counts().sort_index()}")

CELDA 2: Preprocesamiento Estándar (Repaso Eje I)

Descripción: Aplica pipeline de limpieza heredado del Eje I: recorte de silencios, normalización de pico, remuestreo a 16kHz y ajuste de duración fija (3.0s). Este paso garantiza que todas las muestras tengan formato uniforme para extracción de features.

In [ ]:
# ⚙️ Parámetros de preprocesamiento estandarizados para audio ambiental
TARGET_SR = 16000   # Hz (balance entre calidad vocal/ambiental y costo computacional)
TARGET_DUR = 3.0    # segundos (ESC-50 original es 5s; 3s captura eventos clave)
TOP_DB = 35         # Umbral dB para recorte inteligente de silencios/borde

# 🔄 Define función de pipeline reutilizable
def preprocess_audio(audio, sr):
    """Aplica trim, normalización, remuestreo y ajuste de duración."""
    # 📋 Guarda metadatos originales para auditoría
    meta = {"sr_orig": sr, "dur_orig": len(audio)/sr, "amp_orig": np.max(np.abs(audio))}

    # ✂️ Recorta silencios en extremos usando umbral dB
    audio_trim, _ = librosa.effects.trim(audio, top_db=TOP_DB)

    # ⚖️ Normaliza amplitud a [-1, +1] (evita división por cero)
    max_amp = np.max(np.abs(audio_trim))
    audio_norm = audio_trim / max_amp if max_amp > 1e-8 else audio_trim

    # 🔄 Remuestrea si la frecuencia original difiere del objetivo
    if sr != TARGET_SR:
        audio_resamp = librosa.resample(audio_norm, orig_sr=sr, target_sr=TARGET_SR)
        sr = TARGET_SR
    else:
        audio_resamp = audio_norm

    # 📏 Ajusta duración exacta mediante crop centrado o padding con ceros
    target_samples = int(TARGET_DUR * sr)
    if len(audio_resamp) > target_samples:
        # Crop centrado para preservar evento principal
        start = (len(audio_resamp) - target_samples) // 2
        audio_final = audio_resamp[start : start + target_samples]
        meta["ajuste"] = "crop"
    else:
        # Pad al final con silencios
        pad_width = target_samples - len(audio_resamp)
        audio_final = np.pad(audio_resamp, (0, pad_width), mode="constant")
        meta["ajuste"] = "pad"

    # 📊 Actualiza metadatos con valores finales y retorna
    meta.update({"sr_final": sr, "dur_final": len(audio_final)/sr, "amp_final": np.max(np.abs(audio_final))})
    return audio_final, sr, meta

# 🔄 Aplica pipeline a todo el dataset y acumula resultados
processed_audios, processed_srs, meta_list = [], [], []
for idx, row in df_raw.iterrows():
    a, s, m = preprocess_audio(row["audio"], row["sr"])
    processed_audios.append(a)
    processed_srs.append(s)
    m["label"] = row["label"]
    meta_list.append(m)

# 📦 Convierte resultados a DataFrame final listo para feature extraction
df_proc = pd.DataFrame({"audio": processed_audios, "sr": processed_srs, "label": df_raw["label"]})
df_meta = pd.DataFrame(meta_list)

print("✅ Preprocesamiento completado.")
print(f"📏 Audios estandarizados: {df_meta['dur_final'].unique()[0]}s | {df_meta['sr_final'].unique()[0]}Hz")

# 📈 Visualización del preprocesamiento para un audio de ejemplo
sample_idx = 0 # Escoge el primer audio como ejemplo

plt.figure(figsize=(14, 6))

# Subplot 1: Audio Original
plt.subplot(2, 1, 1)
librosa.display.waveshow(y=df_raw['audio'][sample_idx], sr=df_raw['sr'][sample_idx])
plt.title(f'Audio Original ({df_raw["label"][sample_idx]}) - SR: {df_raw["sr"][sample_idx]}Hz')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')

# Subplot 2: Audio Preprocesado
plt.subplot(2, 1, 2)
librosa.display.waveshow(y=df_proc['audio'][sample_idx], sr=df_proc['sr'][sample_idx], color='red')
plt.title(f'Audio Preprocesado - SR: {df_proc["sr"][sample_idx]}Hz, Duración: {df_meta["dur_final"][sample_idx]}s')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud Normalizada')

plt.tight_layout()
plt.show()

CELDA 3: Feature Engineering (Eje II - Núcleo)

Descripción: Extrae características acústicas y cepstrales por muestra. Combina MFCCs estáticos (13) + Deltas (13) + Delta-Deltas (13) + Features espectrales (4). Agrega media y desviación estándar temporal para obtener un vector fijo de 84 dimensiones por audio, compatible con clasificadores clásicos.

In [ ]:
# ️ Función de extracción de features vectorizadas
def extract_features_vector(audio, sr):
    """Extrae MFCCs + Deltas + Features espectrales y colapsa a vector 1D fijo."""
    n_fft_val = 512
    hop_length_val = 256

    # 🔹 MFCCs estáticos (13 coeficientes)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, n_fft=n_fft_val, hop_length=hop_length_val)
    n_frames = mfccs.shape[1] # Referencia del número de frames

    # 🔹 Delta (velocidad de cambio espectral)
    mfccs_delta = librosa.feature.delta(mfccs)
    mfccs_delta = librosa.util.fix_length(mfccs_delta, size=n_frames, axis=1) # Asegura la misma longitud

    # 🔹 Delta-Delta (aceleración del cambio espectral)
    mfccs_delta2 = librosa.feature.delta(mfccs, order=2)
    mfccs_delta2 = librosa.util.fix_length(mfccs_delta2, size=n_frames, axis=1) # Asegura la misma longitud

    # 🔹 Features espectrales adicionales, asegura que sean 2D con el mismo número de frames
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr, n_fft=n_fft_val, hop_length=hop_length_val)[0]
    centroid = librosa.util.fix_length(centroid, size=n_frames).reshape(1, -1)

    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, n_fft=n_fft_val, hop_length=hop_length_val)[0]
    rolloff = librosa.util.fix_length(rolloff, size=n_frames).reshape(1, -1)

    zcr = librosa.feature.zero_crossing_rate(audio, hop_length=hop_length_val, frame_length=n_fft_val)[0]
    zcr = librosa.util.fix_length(zcr, size=n_frames).reshape(1, -1)

    rms = librosa.feature.rms(y=audio, frame_length=n_fft_val, hop_length=hop_length_val)[0]
    rms = librosa.util.fix_length(rms, size=n_frames).reshape(1, -1)

    # 📦 Concatena todas las matrices (13+13+13+1+1+1+1 = 43 filas × N frames)
    all_features = np.vstack([mfccs, mfccs_delta, mfccs_delta2, centroid, rolloff, zcr, rms])

    # 📉 Colapsa dimensión temporal: media y std por característica
    feat_mean = np.mean(all_features, axis=1)
    feat_std = np.std(all_features, axis=1)

    # 🔗 Vector final fijo: 43 medias + 43 stds = 86 características
    return np.hstack([feat_mean, feat_std])

# 🔄 Aplica extracción a todo el dataset procesado
print("🔄 Extrayendo features para todo el dataset...")
feature_vectors = []
for _, row in df_proc.iterrows():
    vec = extract_features_vector(row["audio"], row["sr"])
    feature_vectors.append(vec)

# 📊 Convierte a estructura ML-ready
X = np.array(feature_vectors)          # Matriz (n_muestras, 86)
y = df_proc["label"].values            # Vector de etiquetas
feature_names = [f"F{i}_μ" for i in range(43)] + [f"F{i}_σ" for i in range(43)]

print(f"✅ Dataset listo para ML: {X.shape[0]} muestras × {X.shape[1]} características")
print(f"🏷️ Clases: {np.unique(y)}")

# 📈 Visualización de MFCCs para un audio de ejemplo
sample_audio_mfccs = librosa.feature.mfcc(y=df_proc['audio'][sample_idx], sr=df_proc['sr'][sample_idx], n_mfcc=13, n_fft=512, hop_length=256)

plt.figure(figsize=(10, 4))
librosa.display.specshow(librosa.power_to_db(sample_audio_mfccs, ref=np.max), x_axis='time', sr=df_proc['sr'][sample_idx], hop_length=256)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-frequency Cepstral Coefficients (MFCCs) para audio de ejemplo')
plt.xlabel('Tiempo (s)')
plt.ylabel('Coeficientes MFCC')
plt.tight_layout()
plt.show()

CELDA 4: Entrenamiento y Validación de Modelos

Descripción: Divide datos estratificadamente (80/20), escala features, entrena Random Forest y SVM con validación cruzada, y busca hiperparámetros óptimos. Muestra métricas de validación cruzada para estimar capacidad de generalización.

In [ ]:
# 🔀 División estratificada train/test (80% entreno, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"📊 División: {len(X_train)} train | {len(X_test)} test")

# ⚖️ Escalado de features (crucial para SVM y beneficioso para RF)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 🌲 Entrenamiento Random Forest con validación cruzada
print("\n🌲 Entrenando Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_cv = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
rf_model.fit(X_train_scaled, y_train)
print(f"   CV Accuracy (5-fold): {rf_cv.mean():.3f} ± {rf_cv.std():.3f}")

# 📐 Entrenamiento SVM con kernel RBF
print("\n📐 Entrenando SVM (RBF)...")
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svm_cv = cross_val_score(svm_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
svm_model.fit(X_train_scaled, y_train)
print(f"   CV Accuracy (5-fold): {svm_cv.mean():.3f} ± {svm_cv.std():.3f}")

# 🏆 Selección de mejor modelo para evaluación final
best_model = rf_model if rf_cv.mean() >= svm_cv.mean() else svm_model
best_name = "Random Forest" if best_model is rf_model else "SVM"
print(f"\n✅ Modelo seleccionado para evaluación: {best_name}")

# 📈 Visualización de los resultados de CV
model_names = ['Random Forest', 'SVM']
mean_accuracies = [rf_cv.mean(), svm_cv.mean()]
std_accuracies = [rf_cv.std(), svm_cv.std()]

plt.figure(figsize=(8, 5))
plt.bar(model_names, mean_accuracies, yerr=std_accuracies, capsize=5, color=['forestgreen', 'darkblue'])
plt.ylabel('Precisión Media de CV')
plt.title('Comparación de Precisión de Validación Cruzada (5-fold)')
plt.ylim(0.5, 1.0) # Ajustar el rango del eje Y para mejor visualización
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i, acc in enumerate(mean_accuracies):
    plt.text(i, acc + 0.02, f'{acc:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

CELDA 5: Evaluación, Métricas y Matriz de Confusión

Descripción: Evalúa el modelo seleccionado sobre el conjunto de test, genera reporte de clasificación detallado (precision, recall, f1), visualiza matriz de confusión y analiza errores sistemáticos por clase.

In [ ]:
# 🔮 Predicciones sobre conjunto de test
y_pred = best_model.predict(X_test_scaled)

# 📊 Métricas globales
acc = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy en Test: {acc:.2%}")

# 📋 Reporte detallado por clase
print("\n📋 REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred, target_names=sorted(np.unique(y))))

# 🔥 Matriz de confusión visual
cm = confusion_matrix(y_test, y_pred, labels=sorted(np.unique(y)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=sorted(np.unique(y)),
            yticklabels=sorted(np.unique(y)), cbar=False)
plt.title(f"Matriz de Confusión - {best_name}")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta Real")
plt.tight_layout()
plt.show()

# 🔍 Análisis de errores
errors = np.where(y_pred != y_test)[0]
if len(errors) > 0:
    print(f"\n⚠️ Errores detectados: {len(errors)} muestras mal clasificadas")
    print("Ejemplos de confusiones:")
    for i in errors[:3]:
        print(f"   Real: {y_test[i]} → Predicho: {y_pred[i]}")
else:
    print("\n✅ ¡Clasificación perfecta en el conjunto de test!")

CELDA 6: Interpretación del Modelo y Feature Importance

Descripción: Extrae y visualiza la importancia de características del modelo (válido para RF). Identifica qué coeficientes MFCCs y features espectrales contribuyen más a la decisión, cerrando el ciclo entre ingeniería de features y interpretabilidad.

In [ ]:
# 🌲 Extrae importancia de features (solo aplicable a Random Forest)
if best_name == "Random Forest":
    importances = best_model.feature_importances_
    # Ordena índices por importancia descendente
    top_idx = np.argsort(importances)[::-1][:15]

    plt.figure(figsize=(8, 5))
    plt.barh([feature_names[i] for i in top_idx], importances[top_idx], color='teal', alpha=0.8)
    plt.title("Top 15 Características Más Discriminativas")
    plt.xlabel("Importancia Relativa")
    plt.gca().invert_yaxis()  # Invierte eje Y para lectura descendente
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    # 💡 Interpretación pedagógica
    print("💡 INTERPRETACIÓN:")
    print("   • Las primeras posiciones suelen ser MFCCs 1-3 (forma del tracto vocal/entorno)")
    print("   • Deltas altos indican que la DINÁMICA temporal es clave para distinguir eventos")
    print("   • Features espectrales (centroid/rolloff) aportan contexto de 'brillo' sonoro")
else:
    print("ℹ️ La importancia de features solo se extrajo para Random Forest.")
    print("   Para SVM, se recomienda análisis de SHAP o permutación.")

CELDA 7: Actividades de Consolidación (Alumno)

Descripción: Guía de experimentación, análisis crítico y documentación. El alumno modifica hiperparámetros, evalúa robustez y redacta conclusiones técnicas en celdas Markdown adjuntas.

In [ ]:
print("📝 ACTIVIDADES DE CIERRE - EVIDENCIA 2")
print("="*60)

# 1. Experimentación con escalado y modelos
print("1️⃣ ¿Qué ocurre si eliminas StandardScaler antes de SVM?")
print("   • Ejecuta: svm_no_scale = SVC(kernel='rbf').fit(X_train, y_train)")
print("   • Compara accuracy y tiempos de convergencia.")

# 2. Análisis de robustez
print("\n2️⃣ Agrega ruido blanco al 20% del conjunto de test y re-evalúa.")
print("   • ¿Qué modelo es más resiliente: RF o SVM? ¿Por qué?")

# 3. Feature Selection
print("\n3️⃣ Reduce el vector de 86 a 20 features usando SelectKBest.")
print("   • ¿Se mantiene el accuracy? ¿Qué features sobrevivieron?")

# 📓 Plantilla de bitácora (Copiar a celda Markdown y completar)
bitacora_template = """
## 📓 BITÁCORA EVIDENCIA 2 - EJE II

### 🔍 Hallazgos Técnicos
- **Mejor modelo:** ______ | Accuracy: ______ | CV: ______
- **Features más importantes:** ______, ______, ______
- **Clase más confundida:** ______ → ______

### ⚙️ Decisiones de Diseño
- ¿Por qué elegiste `n_mfcc=13` y `hop_length=256`?
- ¿Cómo justificas el crop centrado vs padding?

### 🤔 Reflexión Crítica
¿Los sonidos ambientales comparten "fonemas" como la voz humana? ¿Cómo afecta esto a la elección de features?
> _Escribe tu respuesta aquí..._
"""
print("\n📋 Plantilla de bitácora copiada. Completa en una celda Markdown nueva.")

CELDA 8: Cierre, Exportación y Checklist

Descripción: Resumen de competencias adquiridas, instrucciones de entrega, checklist de autoevaluación y puente hacia arquitecturas profundas (Eje III).

In [ ]:
# 🎓 Resumen de competencias
print("🎓 CIERRE EVIDENCIA 2 - EJE II")
print("="*60)
print("✅ LO QUE DOMINAS AHORA:")
print("   • Pipeline completo: Audio → Features → Modelo → Métricas")
print("   • Diferencias prácticas entre Random Forest y SVM en audio")
print("   • Importancia del escalado, validación cruzada y estratificación")
print("   • Interpretación de matrices de confusión y feature importance")
print("   • Documentación técnica reproducible en Colab")

# 💾 Instrucciones de entrega
print("\n💾 ENTREGA:")
print("1. Archivo → Descargar → .ipynb")
print("2. Archivo → Descargar → PDF")
print("3. Adjuntar bitácora (Celda 7) y capturas de métricas")
print("4. Nombrar: Evidencia2_EjeII_Apellido_Nombre.ipynb")

# 🔮 Puente al Eje III
print("\n🔮 PRÓXIMO: EJE III - ARQUITECTURAS AVANZADAS")
print("   • Reemplazarás features manuales por aprendizaje automático profundo")
print("   • Espectrogramas → CNNs | Secuencias → RNNs/Transformers")
print("   • Fine-tuning de modelos pre-entrenados (Wav2Vec, Whisper)")

# ✅ Checklist final
print("\n📋 CHECKLIST:")
for item in ["Ejecuté todo en orden", "Comprendí cada transformación", "Comparé RF vs SVM", "Interpreté errores", "Completé bitácora"]:
    print(f"   [ ] {item}")
print("\n🚀 ¡Evidencia completada! Tu pipeline está listo para escalar a Deep Learning.")

## Actividad 1: ¿Qué ocurre si eliminas `StandardScaler` antes de SVM?


In [ ]:
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

print("1️⃣ Probamos: SVM sin StandardScaler")

# Entrenar SVM sin escalado
svm_no_scale = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)

start_time_no_scale = time.time()
svm_no_scale.fit(X_train, y_train)
end_time_no_scale = time.time()
training_time_no_scale = end_time_no_scale - start_time_no_scale

# Predicciones y Accuracy
y_pred_no_scale = svm_no_scale.predict(X_test)
acc_no_scale = accuracy_score(y_test, y_pred_no_scale)

print(f"   • Accuracy (sin escalado): {acc_no_scale:.2%}")
print(f"   • Tiempo de entrenamiento (sin escalado): {training_time_no_scale:.4f} segundos")

# Recuperar datos del SVM con escalado para comparación
# Asegúrate de que svm_model y svm_cv (de CELDA 4) estén disponibles en el entorno
# Para mayor claridad, volvemos a calcular el tiempo de entrenamiento del modelo escalado
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42) # Re-instanciar para medir tiempo
start_time_scaled = time.time()
svm_model.fit(X_train_scaled, y_train)
end_time_scaled = time.time()
training_time_scaled = end_time_scaled - start_time_scaled

print(f"\n   • Accuracy (con escalado): {svm_cv.mean():.2%} (CV) / {accuracy_score(y_test, svm_model.predict(X_test_scaled)):.2%} (Test)")
print(f"   • Tiempo de entrenamiento (con escalado): {training_time_scaled:.4f} segundos")


## 💡 **Análisis de la Actividad 1:**
   Al eliminar `StandardScaler`, la precisión del modelo SVM disminuye significativamente, lo que demuestra la importancia del escalado para este tipo de modelo.
   El tiempo de entrenamiento sin escalado es comparable o incluso ligeramente menor. Esto puede deberse a la naturaleza de los datos o al tamaño reducido del dataset.

**Accuracy:**
Al no aplicar el escalado de características (StandardScaler), la precisión del modelo SVM en el conjunto de test disminuyó significativamente de aproximadamente 91.67% (con escalado) a 83.33% (sin escalado). Esto subraya la importancia crítica del escalado para los modelos SVM. Estos son sensibles a la escala de las características, y sin normalización, las características con rangos más grandes pueden dominar la función de distancia, afectando negativamente el rendimiento.

**Tiempo de Entrenamiento:**
El tiempo de entrenamiento disminuyó ligeramente de 0.0101 segundos (con escalado) a 0.0073 segundos (sin escalado). Si bien en datasets más grandes el entrenamiento sin escalado puede ser más lento debido a la dificultad del optimizador para converger, en este caso, la diferencia es mínima e incluso se observa una ligera reducción, lo que podría deberse a la simplicidad del dataset.

## Actividad 2: Robustez ante Ruido Blanco


In [ ]:
import random

print("2️⃣ Probamos: Robustez ante Ruido Blanco")

# Función para añadir ruido blanco
def add_white_noise(data, noise_factor=0.01):
    noise = np.random.normal(0, noise_factor, data.shape)
    return data + noise

# Identificar el 20% de los índices del conjunto de test
num_test_samples = X_test_scaled.shape[0]
num_noisy_samples = int(num_test_samples * 0.2)

# Asegurarse de que no haya más índices seleccionados que el tamaño del conjunto de test
if num_noisy_samples == 0 and num_test_samples > 0:
    num_noisy_samples = 1 # Asegurar al menos una muestra si el test set no es vacío

# Seleccionar índices aleatorios para aplicar el ruido
noisy_indices = random.sample(range(num_test_samples), num_noisy_samples)

# Crear una copia del conjunto de test escalado para no modificar el original
X_test_noisy = X_test_scaled.copy()

# Aplicar ruido blanco a las muestras seleccionadas
for idx in noisy_indices:
    X_test_noisy[idx] = add_white_noise(X_test_noisy[idx])

print(f"   • Ruido blanco añadido a {num_noisy_samples} de {num_test_samples} muestras del conjunto de test.")

# Re-evaluar Random Forest en el conjunto de test con ruido
y_pred_rf_noisy = rf_model.predict(X_test_noisy)
acc_rf_noisy = accuracy_score(y_test, y_pred_rf_noisy)
print(f"   • Accuracy de Random Forest (con ruido): {acc_rf_noisy:.2%}")

# Re-evaluar SVM en el conjunto de test con ruido
y_pred_svm_noisy = svm_model.predict(X_test_noisy)
acc_svm_noisy = accuracy_score(y_test, y_pred_svm_noisy)
print(f"   • Accuracy de SVM (con ruido): {acc_svm_noisy:.2%}")


## 💡 **Análisis de la Actividad 2:**

 Ambos modelos (Random Forest y SVM) mostraron una resiliencia similar al ruido blanco. Las precisiones se mantuvieron idénticas al 91.67%, lo que sugiere que para la cantidad y el tipo de ruido aplicado, ninguno de los modelos se vio significativamente más afectado que el otro.

*   Es posible que el factor de ruido aplicado (noise_factor=0.01) no haya sido lo suficientemente alto como para mostrar una divergencia clara.

¿Por qué podrían tener una resiliencia similar?

1. **Random Forest**: Generalmente, los Random Forests son conocidos por su robustez al ruido y al sobreajuste debido a su naturaleza de ensemble. Al promediar las predicciones de múltiples árboles de decisión entrenados en diferentes subconjuntos de datos y características, el impacto de una pequeña perturbación en algunas muestras tiende a ser mitigado. Si un árbol se ve afectado por el ruido, los demás árboles no afectados pueden compensarlo.

2.   **SVM**: Los SVM, especialmente con kernels no lineales como el RBF, buscan un hiperplano que maximice el margen entre las clases. Si bien pueden ser sensibles a datos ruidosos o atípicos (outliers) que caen dentro del margen o son vectores de soporte, la robustez también puede depender de la complejidad de la separación de clases y de los parámetros C y gamma. En este caso, la configuración y el ruido añadido no fueron suficientes para desestabilizar significativamente su capacidad de clasificación.

## Actividad 3: Reducción de Dimensionalidad con `SelectKBest`


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

print("3️⃣ Experimentando: Reducción de Features con SelectKBest")

# 📏 Inicializar SelectKBest para seleccionar las 20 mejores características usando f_classif
selector = SelectKBest(score_func=f_classif, k=20)

# 🚀 Ajustar el selector a los datos de entrenamiento escalados y transformar
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print(f"   • Número original de características: {X_train_scaled.shape[1]}")
print(f"   • Número de características seleccionadas: {X_train_selected.shape[1]}")

# 📊 Obtener los índices de las características seleccionadas
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = [feature_names[i] for i in selected_feature_indices]
print(f"   • Características seleccionadas: {', '.join(selected_feature_names)}")

# 🌲 Re-entrenar Random Forest con las características seleccionadas
print("\n   🌲 Re-entrenando Random Forest con features seleccionadas...")
rf_model_selected = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_model_selected.fit(X_train_selected, y_train)
y_pred_rf_selected = rf_model_selected.predict(X_test_selected)
acc_rf_selected = accuracy_score(y_test, y_pred_rf_selected)
print(f"   • Accuracy de Random Forest (20 features): {acc_rf_selected:.2%}")

# 📐 Re-entrenar SVM con las características seleccionadas
print("\n   📐 Re-entrenando SVM (RBF) con features seleccionadas...")
svm_model_selected = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=42)
svm_model_selected.fit(X_train_selected, y_train)
y_pred_svm_selected = svm_model_selected.predict(X_test_selected)
acc_svm_selected = accuracy_score(y_test, y_pred_svm_selected)
print(f"   • Accuracy de SVM (20 features): {acc_svm_selected:.2%}")

print("\n💡 **Análisis de la Actividad 3:**")
print("   **Comparación de Accuracy:**")
print(f"   • Accuracy RF (original): {acc_rf_selected:.2%} (utilizando el mismo resultado de test anterior, ya que el modelo RF original fue entrenado con todas las features)")
print(f"   • Accuracy SVM (original): {acc_svm_selected:.2%} (utilizando el mismo resultado de test anterior, ya que el modelo SVM original fue entrenado con todas las features)")

# Aquí se usarán los valores de accuracy del kernel para los modelos originales
print(f"   • Accuracy RF (original): {accuracy_score(y_test, rf_model.predict(X_test_scaled)):.2%}")
print(f"   • Accuracy SVM (original): {accuracy_score(y_test, svm_model.predict(X_test_scaled)):.2%}")

if acc_rf_selected >= accuracy_score(y_test, rf_model.predict(X_test_scaled)) and acc_svm_selected >= accuracy_score(y_test, svm_model.predict(X_test_scaled)):
    print("   La precisión se mantuvo o incluso mejoró ligeramente con la reducción de características, lo que sugiere que muchas de las características originales eran redundantes o no informativas. Esto también puede ayudar a reducir el sobreajuste.")
elif acc_rf_selected < accuracy_score(y_test, rf_model.predict(X_test_scaled)) or acc_svm_selected < accuracy_score(y_test, svm_model.predict(X_test_scaled)):
    print("   La precisión disminuyó con la reducción de características, lo que indica que algunas características importantes fueron descartadas. Sin embargo, la reducción de dimensionalidad puede ser útil para la velocidad de entrenamiento o para modelos que sufren de la 'maldición de la dimensionalidad'.")
else:
    print("   No hubo un cambio significativo en la precisión, lo cual es positivo ya que hemos reducido la complejidad del modelo sin sacrificar rendimiento.")

print("\n   **Features seleccionadas:**")
print("   Las características seleccionadas son aquellas que `SelectKBest` consideró más relevantes para discriminar entre las clases. A menudo, estas son las que tienen mayor varianza o una correlación más fuerte con la variable objetivo.")


## 💡 **Análisis de la Actividad 3:**


Se **mantiene el accuracy**, la precisión de ambos modelos (Random Forest y SVM) se mantuvo o incluso mejoró ligeramente con la reducción de características:

**Random Forest**: La precisión pasó de un 91.67% (con todas las características) a un 91.67% (con 20 características).

**SVM**: La precisión mejoró de un 91.67% (con todas las características) a un 95.83% (con 20 características).

Resulta Positivo, ya que muchas de las 86 características originales eran redundantes o no informativas para la tarea de clasificación. Al eliminar estas características menos relevantes, no solo se simplifican los modelos y se reduce el costo computacional, sino que en el caso del SVM, incluso se observa una ligera mejora en el rendimiento.


**En Conclusión**: La precisión de ambos modelos se mantuvo o incluso mejoró ligeramente con la reducción de características. Esto sugiere que muchas de las características originales eran redundantes o no aportaban información significativa, y que la selección de características fue efectiva. Reducir la dimensionalidad puede ayudar a simplificar el modelo y, en algunos casos, a prevenir el sobreajuste.

Las características seleccionadas (F0_μ, F1_μ, etc.) son aquellas que SelectKBest identificó como las más relevantes para diferenciar entre las clases, basándose en la prueba estadística F (f_classif).